# 05 MVCC

## 八股问答

### 1. MVCC 是什么？解决了什么问题？

MVCC（多版本并发控制）让读操作不阻塞写、写操作不阻塞读。实现方式是每行保留
多个历史版本，事务通过 Read View 判断哪个版本对自己可见，普通 SELECT 完全不
加锁，并发能力大幅提升。

### 2. MVCC 是如何实现的？

三个组成部分：

- 隐藏列：每行有 `DB_TRX_ID`（最近修改该行的事务 ID）和 `DB_ROLL_PTR`
  （回滚指针）；没有主键时才有 `DB_ROW_ID`。
- undo log 版本链：每次修改把旧版本写入 undo log，新版本通过回滚指针指向
  旧版本，串成版本链。
- Read View：事务做快照读时记录“哪些事务还没提交”，用来判断版本链上
  哪个版本可见。

### 3. 什么是快照读和当前读？

| 类型 | 语句 | 读的版本 | 加锁 |
| --- | --- | --- | --- |
| 快照读 | 普通 `SELECT` | MVCC 可见版本 | 不加锁 |
| 当前读 | `SELECT ... FOR UPDATE`、`SELECT ... FOR SHARE`、`UPDATE`、`DELETE` | 最新版本 | 加锁 |

当前读必须读最新版本并加锁，否则并发写会互相覆盖。

### 4. Read View 的结构是什么？

Read View 是事务生成快照时记录的“哪些事务还没提交”的视图，包含四个字段：

- `m_ids`：生成时仍未提交的事务 ID 列表。
- `min_trx_id`：列表中最小事务 ID。
- `max_trx_id`：下一个将要分配的事务 ID。
- `creator_trx_id`：生成该 Read View 的事务自己的 ID。

### 5. Read View 的可见性规则？

沿版本链从新到旧判断，任一版本满足下列条件即可见：

1. 版本的事务 ID 等于 `creator_trx_id`：自己改的，可见。
2. 小于 `min_trx_id`：已经提交，可见。
3. 大于等于 `max_trx_id`：生成快照之后才启动的事务，不可见。
4. 在 `m_ids` 中：还没提交，不可见；不在 `m_ids` 中：已提交，可见。

### 6. Read View 如何解决脏读、不可重复读、幻读？

- 脏读：未提交事务的版本 ID 在 `m_ids` 里，对别的快照读不可见，所以读不到
  未提交数据。
- 不可重复读：同一事务复用同一个 Read View，另一次读仍按同一快照判断，
  两次读同一行结果一致。
- 幻读：快照读层面，新插入记录的事务 ID 大于等于 `max_trx_id`，不可见，
  行数不变化。

注意：快照读只是“看不见”幻读；真正阻止别人插入的，是当前读时加的间隙锁，
两者在 RR 下共同生效。

### 7. RC 和 RR 下 MVCC 的区别？

- READ COMMITTED：每次普通 SELECT 都生成新的 Read View，所以能看到其他
  事务新提交的修改，也因此存在不可重复读。
- REPEATABLE READ：只在事务第一次读（快照读或当前读）时生成 Read View，
  之后一直复用，所以快照读结果稳定。

### 8. RR 下执行 UPDATE 会发生什么？

UPDATE 是当前读，不受 Read View 可见性限制：

1. 先按条件定位记录，加 X 锁（RR 下是临键锁）。
2. 读取最新已提交版本。
3. 写 undo log 保存旧版本，把当前版本标记为历史版本。
4. 生成新版本，`DB_TRX_ID` 更新为当前事务。
5. 事务提交后新版本对其他事务可见，回滚则沿 undo 版本链恢复。

这也解释了经典问题：RR 下两个事务同时 UPDATE 同一行不会“等对方提交”后直接
覆盖，而是后执行者基于最新版本再更新；如果先快照读后 UPDATE，当前读依然
读到最新值而不是快照值。

### 9. 回滚的流程是什么？

回滚时事务按 undo log 中自己产生的版本反向执行：把每一处修改恢复成上一个
版本，删除插入的记录，恢复删除的记录。由于版本链按事务串行记录，回滚不需要
扫描整张表。MVCC 的快照读和事务回滚共用同一条 undo 版本链。


## 面试话术

“MVCC 三件套：隐藏列、undo 版本链、Read View。快照读沿版本链找第一个满足
可见性规则的版本，不加锁；当前读读最新版本并加锁。RC 每次读都生成新
Read View，RR 只在第一次读生成，所以 RC 有不可重复读、RR 没有。UPDATE 走
当前读，所以即使先做了快照读，修改也基于最新版本，这也解释了 RR 下并发
更新的行为。”


## 自查清单

- [ ] 能画出版本链：隐藏列、undo、回滚指针的关系
- [ ] 能默写 Read View 四个字段和可见性判断规则
- [ ] 能说清 RC/RR 生成 Read View 的时机差异
- [ ] 能在双终端演示“先快照读再 UPDATE 基于最新值”
- [ ] 能解释回滚为什么依赖 undo log
